# A5B: CNN Experiments - Pipeline de Experimentação

Notebook para testar diferentes configurações de hiperparâmetros usando `ExperimentRunner`.

Cada experimento é salvo com:
- Modelo treinado (.h5)
- Histórico de treinamento (loss/accuracy)
- Cópia da configuração usada (para rastreabilidade)

## 1. Importações e Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Adicionar src ao path (relativo - funciona em qualquer lugar)
notebook_dir = Path.cwd()
src_path = notebook_dir / "src" if (notebook_dir / "src").exists() else notebook_dir.parent / "src"
sys.path.insert(0, str(src_path))

# Importar ferramentas de experimentação
from models.experiment_runner import ExperimentRunner, run_multiple_experiments
from models.cnn_config import list_available_configs, load_config

print("✓ Importações realizadas com sucesso!")

I0000 00:00:1773154720.972196   86832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773154720.972468   86832 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773154720.998154   86832 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ Importações realizadas com sucesso!


I0000 00:00:1773154721.553895   86832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773154721.554048   86832 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. Listar Configurações Disponíveis

In [18]:
# Ver todas as configurações disponíveis
available_configs = list_available_configs()

print("\nConfiguções disponíveis para experimentação:\n")
for i, config_name in enumerate(available_configs, 1):
    print(f"  {i}. {config_name}")
    
    # Carregar e exibir parâmetros principais
    config = load_config(config_name)
    model_cfg = config.get("model", {})
    train_cfg = config.get("training", {})
    
    print(f"     Filtros: {model_cfg.get('filters')}")
    print(f"     L2: {model_cfg.get('l2_regularizer')}")
    print(f"     Conv Dropout: {model_cfg.get('conv_dropout_rate')}")
    print(f"     Dense Dropout: {model_cfg.get('dense_dropout_rate')}")
    print(f"     Learning Rate: {train_cfg.get('learning_rate')}")
    print()


Configuções disponíveis para experimentação:

  1. baseline
     Filtros: [32, 64]
     L2: 0.001
     Conv Dropout: 0.2
     Dense Dropout: 0.5
     Learning Rate: 0.001

  2. higher_dropout
     Filtros: [32, 64]
     L2: 0.001
     Conv Dropout: 0.3
     Dense Dropout: 0.6
     Learning Rate: 0.0005

  3. model_architechture
     Filtros: [32, 64]
     L2: 0.001
     Conv Dropout: 0.2
     Dense Dropout: 0.5
     Learning Rate: 0.001



## 3. Rodar Um Experimento

In [19]:
# Teste rápido com config baseline
print("\nRodando experimento model architecture...\n")

runner = ExperimentRunner("model_architechture")
result_baseline = runner.run_full_pipeline(
    limit_samples=10,  # Usar todos os dados (ou 50 para teste rápido)
    verbose=1
)

print(f"\nExperimento salvo em: {result_baseline['experiment_dir']}")


Rodando experimento model architecture...

✓ ExperimentRunner inicializado com config: model_architechture
  Configurações disponíveis: ['baseline', 'higher_dropout', 'model_architechture']

EXPERIMENTO: model_architechture

📂 Carregando dados...
✓ Dados carregados: (10, 128, 128, 9), labels: (10,)
  Classes: [0 1], distribuição: [6 4]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: model_architechture_20260310_125114

🚀 Iniciando treinamento...
  Batch size: 32
  Epochs: 50
  Learning rate: 0.001
Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 669ms/step - accuracy: 0.3750 - loss: 1.2179 - val_accuracy: 1.0000 - val_loss: 0.2913
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.7500 - loss: 6.2463 - val_accuracy: 0.5000 - val_loss: 1.8989
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.6250 - loss: 8.8929 - val_accuracy: 0.5000 - val_loss: 4.8767
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.6250 - loss: 8.2452 - val_accuracy: 0.5000 

✓ Treinamento concluído!
✓ Modelo salvo: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125114/model.h5
✓ Histórico salvo: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125114/history.json
✓ Resultado adicionado ao log: /home/cc09-g1/g01/outputs/trained_models/experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: model_architechture
  experiment_dir: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125114
  timestamp: 2026-03-10 12:51:19
  final_train_loss: 0.6566
  final_train_acc: 0.8750
  final_val_loss: 0.5353
  final_val_acc: 1.0000
  epochs_run: 50

Experimento salvo em: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125114


## 4. Rodar Múltiplos Experimentos e Comparar

In [20]:
# Rodar múltiplas configurações sequencialmente
print("\nRodando múltiplos experimentos...\n")

results = run_multiple_experiments(
    ["baseline", "higher_dropout" ,"model_architechture"],  # Configurações a serem testadas
    limit_samples=10  # Todos os dados (ou 50 para teste rápido)
)

print("\nExperimentos concluídos!")


Rodando múltiplos experimentos...

✓ ExperimentRunner inicializado com config: baseline
  Configurações disponíveis: ['baseline', 'higher_dropout', 'model_architechture']

EXPERIMENTO: baseline

📂 Carregando dados...


✓ Dados carregados: (10, 128, 128, 9), labels: (10,)
  Classes: [0 1], distribuição: [6 4]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: baseline_20260310_125146

🚀 Iniciando treinamento...
  Batch size: 32
  Epochs: 50
  Learning rate: 0.001
Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 670ms/step - accuracy: 0.3750 - loss: 1.7984 - val_accuracy: 0.5000 - val_loss: 1.0171
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.6250 - loss: 14.4762 - val_accuracy: 1.0000 - val_loss: 0.2791
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.6250 - loss: 16.5203 - val_accuracy: 1.0000 - val_loss: 0.2708
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.7500 - loss: 14.4725 - val_accuracy: 1.0000 - val_loss: 0.2662
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.7500 - loss: 12.6868 - val_accuracy: 1.0000 - val_loss: 0.2648
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8750 - loss: 2.8613 - val_accuracy: 1.0000 - val

✓ Treinamento concluído!
✓ Modelo salvo: /home/cc09-g1/g01/outputs/trained_models/baseline_20260310_125146/model.h5
✓ Histórico salvo: /home/cc09-g1/g01/outputs/trained_models/baseline_20260310_125146/history.json
✓ Resultado adicionado ao log: /home/cc09-g1/g01/outputs/trained_models/experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: baseline
  experiment_dir: /home/cc09-g1/g01/outputs/trained_models/baseline_20260310_125146
  timestamp: 2026-03-10 12:51:51
  final_train_loss: 0.6981
  final_train_acc: 0.8750
  final_val_loss: 0.4929
  final_val_acc: 1.0000
  epochs_run: 50
✓ ExperimentRunner inicializado com config: higher_dropout
  Configurações disponíveis: ['baseline', 'higher_dropout', 'model_architechture']

EXPERIMENTO: higher_dropout

📂 Carregando dados...
✓ Dados carregados: (10, 128, 128, 9), labels: (10,)
  Classes: [0 1], distribuição: [6 4]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: higher_dropout_20260310_125217

🚀 Iniciando treinamento...
  Ba

✓ Treinamento concluído!
✓ Modelo salvo: /home/cc09-g1/g01/outputs/trained_models/higher_dropout_20260310_125217/model.h5
✓ Histórico salvo: /home/cc09-g1/g01/outputs/trained_models/higher_dropout_20260310_125217/history.json
✓ Resultado adicionado ao log: /home/cc09-g1/g01/outputs/trained_models/experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: higher_dropout
  experiment_dir: /home/cc09-g1/g01/outputs/trained_models/higher_dropout_20260310_125217
  timestamp: 2026-03-10 12:52:22
  final_train_loss: 0.5316
  final_train_acc: 0.8750
  final_val_loss: 0.4974
  final_val_acc: 1.0000
  epochs_run: 50
✓ ExperimentRunner inicializado com config: model_architechture
  Configurações disponíveis: ['baseline', 'higher_dropout', 'model_architechture']

EXPERIMENTO: model_architechture

📂 Carregando dados...
✓ Dados carregados: (10, 128, 128, 9), labels: (10,)
  Classes: [0 1], distribuição: [6 4]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: model_architechture_20260310_

✓ Treinamento concluído!
✓ Modelo salvo: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125249/model.h5
✓ Histórico salvo: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125249/history.json
✓ Resultado adicionado ao log: /home/cc09-g1/g01/outputs/trained_models/experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: model_architechture
  experiment_dir: /home/cc09-g1/g01/outputs/trained_models/model_architechture_20260310_125249
  timestamp: 2026-03-10 12:52:54
  final_train_loss: 0.5927
  final_train_acc: 0.8750
  final_val_loss: 1.0489
  final_val_acc: 0.5000
  epochs_run: 50

Experimentos concluídos!


## 5. Comparar Resultados

In [21]:
# Criar tabela de comparação
comparison_data = []

for r in results:
    if "error" not in r:
        comparison_data.append({
            "Config": r.get("config_name"),
            "Train Loss": f"{r.get('final_train_loss', 0):.4f}",
            "Train Acc": f"{r.get('final_train_acc', 0):.4f}",
            "Val Loss": f"{r.get('final_val_loss', 0):.4f}",
            "Val Acc": f"{r.get('final_val_acc', 0):.4f}",
            "Epochs": r.get("epochs_run"),
        })
    else:
        print(f"❌ Erro em {r.get('config_name')}: {r.get('error')}")

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPARAÇÃO DE EXPERIMENTOS")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)


COMPARAÇÃO DE EXPERIMENTOS
             Config Train Loss Train Acc Val Loss Val Acc  Epochs
           baseline     0.6981    0.8750   0.4929  1.0000      50
     higher_dropout     0.5316    0.8750   0.4974  1.0000      50
model_architechture     0.5927    0.8750   1.0489  0.5000      50


## 6. Inspecionar Resultados de um Experimento

In [26]:
# Carregar histórico de um experimento
import json

# Usar o caminho do primeiro experimento resultante
if results and "experiment_dir" in results[0]:
    exp_dir = Path(results[0]["experiment_dir"])
    
    # Carregar config usada
    config_file = exp_dir / "config_used.json"
    if config_file.exists():
        with open(config_file) as f:
            config_used = json.load(f)
        print(f"\nConfig usada em {exp_dir.name}:")
        print(json.dumps(config_used, indent=2))
    
    # Carregar histórico
    history_file = exp_dir / "history.json"
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
        
        print(f"\nHistórico de treinamento (últimas 5 épocas):")
        epochs = len(history['loss'])
        for i in range(max(0, epochs-5), epochs):
            print(f"  Época {i+1}: Loss={history['loss'][i]:.4f}, Acc={history['accuracy'][i]:.4f}, "
                  f"Val Loss={history['val_loss'][i]:.4f}, Val Acc={history['val_accuracy'][i]:.4f}")


Config usada em baseline_20260305_101434:
{
  "model": {
    "input_shape": [
      128,
      128,
      9
    ],
    "num_classes": 2,
    "filters": [
      32,
      64
    ],
    "kernel_size": 3,
    "pool_size": 2,
    "l2_regularizer": 0.001,
    "conv_dropout_rate": 0.2,
    "dense_dropout_rate": 0.5,
    "dense_units": 128
  },
  "training": {
    "batch_size": 32,
    "epochs": 50,
    "learning_rate": 0.001,
    "validation_split": 0.2,
    "optimizer": "adam"
  },
  "data": {
    "dataset_path": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\pixels_dataset.csv",
    "codes_path": "c:\\Users\\Inteli\\Desktop\\g01\\data\\extracted_codes.json",
    "normalizer_path": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\a04_cnn_data_prep\\cnn_normalizer_zscore.npz",
    "normalization_method": "zscore"
  },
  "output": {
    "models_dir": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\trained_models",
    "logs_dir": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\training_logs",
    "save_model": 

## 7. Histórico de Todos os Experimentos

Visualize o arquivo CSV com todos os experimentos rodados (append automático).

In [11]:
# Carregar e exibir histórico de experimentos
# Usar o mesmo caminho que experiment_runner usa
config = load_config("baseline")
output_cfg = config["output"]
csv_path = Path(output_cfg["models_dir"]) / "experiments_log.csv"

if csv_path.exists():
    df_experiments = pd.read_csv(csv_path)
    
    print("\n" + "="*120)
    print("HISTÓRICO DE TODOS OS EXPERIMENTOS")
    print("="*120)
    print(df_experiments.to_string(index=False))
    print("="*120)
    print(f"\n📊 Total de experimentos: {len(df_experiments)}")
    print(f"📈 Melhor Val Acc: {df_experiments['val_acc'].max():.4f} ({df_experiments.loc[df_experiments['val_acc'].idxmax(), 'config_name']})")
    print(f"📉 Menor Val Loss: {df_experiments['val_loss'].min():.4f} ({df_experiments.loc[df_experiments['val_loss'].idxmin(), 'config_name']})")
else:
    print(f"⚠️ Arquivo de log não encontrado em: {csv_path}")


HISTÓRICO DE TODOS OS EXPERIMENTOS
          timestamp config_name  train_loss  train_acc  val_loss  val_acc  epochs                                                              experiment_dir
2026-03-05 10:18:50    baseline    0.567335      0.875  0.608167      1.0      50 c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260305_101834
2026-03-05 10:25:10    baseline    0.536653      0.875  0.488101      1.0      50 c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260305_102454

📊 Total de experimentos: 2
📈 Melhor Val Acc: 1.0000 (baseline)
📉 Menor Val Loss: 0.4881 (baseline)


## 8. Criar Novas Configurações para Ablations

Para testar novos hiperparâmetros:

1. **Crie um novo arquivo `.yaml`** em `src/models/configs/`
   - Exemplo: `src/models/configs/mais_filtros.yaml`
   
2. **Use `ExperimentRunner`** com o novo nome:
   ```python
   runner = ExperimentRunner("mais_filtros")
   runner.run_full_pipeline()
   ```

3. **Resultados** aparecem automaticamente em:
   ```
   outputs/trained_models/experiments_log.csv
   ```